In [11]:
%pip install pytesseract pillow requests

Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
import re
import requests
from PIL import Image
from io import BytesIO
import pytesseract
from urllib.parse import urljoin

# ==============================
# CẤU HÌNH ĐƯỜNG DẪN TESSERACT
# ==============================
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# File Markdown đầu vào
INPUT_MD_FILE = "DATA_WEB_FIT.md"

# File Markdown đầu ra sau khi thêm OCR
OUTPUT_AUGMENTED_MD_FILE = "DATA_WEB_FIT_TESSERACT.md"

# Website gốc, dùng để nối các link ảnh dạng /Resources/...
BASE_URL = "http://fit.hcmute.edu.vn"

# Test trước vài ảnh để tránh chạy quá lâu
# Khi chạy ổn rồi thì đổi MAX_IMAGES = None
MAX_IMAGES = None

In [13]:
try:
    print("Tesseract version:")
    print(pytesseract.get_tesseract_version())

    languages = pytesseract.get_languages(config="")
    print("\nDanh sách ngôn ngữ đang có:")
    print(languages)

    if "vie" in languages:
        print("\n[OK] Đã có tiếng Việt: vie")
    else:
        print("\n[!] Chưa có tiếng Việt 'vie'.")
        print("Bạn cần copy file vie.traineddata vào thư mục:")
        print(r"C:\Program Files\Tesseract-OCR\tessdata")

except pytesseract.TesseractNotFoundError:
    print("[-] Không tìm thấy Tesseract.")
    print("Kiểm tra lại đường dẫn:")
    print(pytesseract.pytesseract.tesseract_cmd)

except Exception as e:
    print("[-] Lỗi khi kiểm tra Tesseract:")
    print(e)

Tesseract version:
5.5.0.20241111

Danh sách ngôn ngữ đang có:
['eng', 'osd', 'vie']

[OK] Đã có tiếng Việt: vie


In [14]:
def download_image(image_url, timeout=10):
    """
    Tải ảnh từ URL.
    Có timeout để tránh bị treo quá lâu.
    """
    try:
        response = requests.get(
            image_url,
            timeout=timeout,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        response.raise_for_status()

        return response.content

    except requests.exceptions.Timeout:
        print(f"[-] Tải ảnh quá lâu, bỏ qua: {image_url}")
        return None

    except requests.exceptions.RequestException as e:
        print(f"[-] Lỗi khi tải ảnh: {image_url}")
        print(e)
        return None


def extract_text_with_tesseract(image_content, lang="vie+eng", timeout=10):
    """
    Dùng Tesseract OCR để đọc chữ trong ảnh.
    lang='vie+eng' nghĩa là đọc cả tiếng Việt và tiếng Anh.
    timeout=10 để tránh OCR bị treo quá lâu.
    """
    try:
        img = Image.open(BytesIO(image_content)).convert("RGB")

        extracted_text = pytesseract.image_to_string(
            img,
            lang=lang,
            timeout=timeout
        )

        return extracted_text.strip()

    except RuntimeError:
        print("[-] OCR quá lâu, bỏ qua ảnh này.")
        return ""

    except pytesseract.TesseractNotFoundError:
        print("[-] Không tìm thấy phần mềm Tesseract.")
        print("Kiểm tra lại đường dẫn tesseract.exe.")
        return ""

    except Exception as e:
        print("[-] Lỗi khi OCR ảnh:")
        print(e)
        return ""

In [15]:
print(f"Đang đọc file: {INPUT_MD_FILE}")

try:
    with open(INPUT_MD_FILE, "r", encoding="utf-8") as f:
        md_content = f.read()

except FileNotFoundError:
    print(f"[-] Không tìm thấy file: {INPUT_MD_FILE}")
    md_content = ""


# Tìm ảnh dạng Markdown: ![](link)
# Và ảnh dạng HTML: <img src="link">
image_pattern = re.compile(
    r'!\[.*?\]\((.*?)\)|<img[^>]+src=["\'](.*?)["\'][^>]*>',
    re.IGNORECASE
)

matches = list(image_pattern.finditer(md_content))

print(f"Tìm thấy {len(matches)} ảnh trong file Markdown.")

# Lấy danh sách ảnh không trùng
image_items = []
seen_urls = set()

for match in matches:
    full_match_text = match.group(0)

    image_url = match.group(1) if match.group(1) else match.group(2)

    if not image_url:
        continue

    image_url = image_url.strip()

    # Bỏ qua ảnh base64
    if image_url.startswith("data:image"):
        continue

    # Nếu link ảnh là /Resources/... thì nối với domain gốc
    image_url_full = urljoin(BASE_URL, image_url)

    if image_url_full in seen_urls:
        continue

    seen_urls.add(image_url_full)

    image_items.append({
        "full_match_text": full_match_text,
        "image_url": image_url,
        "image_url_full": image_url_full
    })


print(f"Số ảnh sau khi loại trùng: {len(image_items)}")

if MAX_IMAGES is not None:
    image_items = image_items[:MAX_IMAGES]
    print(f"Chỉ xử lý thử {MAX_IMAGES} ảnh đầu tiên.")


augmented_content = md_content

for index, item in enumerate(image_items, start=1):
    full_match_text = item["full_match_text"]
    image_url_full = item["image_url_full"]

    print("=" * 80)
    print(f"Ảnh {index}/{len(image_items)}")
    print(f"Đang xử lý: {image_url_full}")

    image_content = download_image(image_url_full, timeout=10)

    if image_content is None:
        print("[-] Không tải được ảnh, bỏ qua.")
        continue

    ocr_text = extract_text_with_tesseract(
        image_content,
        lang="vie+eng",
        timeout=10
    )

    if not ocr_text:
        print("[!] Không đọc được chữ hoặc ảnh không có chữ.")
        continue

    print("[OK] OCR đọc được:")
    print(ocr_text[:300])

    # Tạo block OCR bằng cách nối chuỗi, tránh lỗi f-string với dấu ```
    ocr_block = (
        "\n\n"
        "<!-- OCR_TEXT_START -->\n"
        "**Nội dung chữ đọc từ ảnh bằng Tesseract:**\n\n"
        "~~~text\n"
        + ocr_text +
        "\n~~~\n"
        "<!-- OCR_TEXT_END -->\n\n"
    )

    # Chèn OCR ngay sau ảnh trong Markdown
    augmented_content = augmented_content.replace(
        full_match_text,
        full_match_text + ocr_block,
        1
    )


with open(OUTPUT_AUGMENTED_MD_FILE, "w", encoding="utf-8") as f:
    f.write(augmented_content)

print("=" * 80)
print("[DONE] Đã tạo file mới:")
print(OUTPUT_AUGMENTED_MD_FILE)

Đang đọc file: DATA_WEB_FIT.md
Tìm thấy 574 ảnh trong file Markdown.
Số ảnh sau khi loại trùng: 391
Ảnh 1/391
Đang xử lý: http://fit.hcmute.edu.vn/Services/GetArticleImage.ashx?option=0&Id=09319281-8d8d-422b-bd6a-a431a9d3fda6
[OK] OCR đọc được:
A

DIKANS
4 lộ
IQ TECHNOLOGY

Bright Minds, Brilliant Solutions

FRESHER SOFTWARE ENGINEER
Wava, .Net, Front-End)
Ảnh 2/391
Đang xử lý: http://fit.hcmute.edu.vn/Services/GetArticleImage.ashx?option=0&Id=ec90fc2f-26ad-44ff-bf7d-cdec3688ad05
[!] Không đọc được chữ hoặc ảnh không có chữ.
Ảnh 3/391
Đang xử lý: http://fit.hcmute.edu.vn/Services/GetArticleImage.ashx?option=0&Id=eeea8be1-e7ea-4ed4-851a-1fa0919e34e5
[OK] OCR đọc được:
KHOA CÔNG NGHỆ THONG TIN

TRUONG DAI HOC CONG NGHE KY THUAT THANH PHO HO CHi MINH =

HOM-UTE

Th
tí†rrrrrt:

> it 500000)
IKHOA CONG NGHE THONG TIN HCM UTE "]
Sử hy

“—ecedct

ở
"sở:
ĐC sử V  < 5Nni
Ảnh 4/391
Đang xử lý: http://fit.hcmute.edu.vn/Resources/ImagesPortal/PhongBan/arrow_all_phong.png
[!] Không đọc được chữ hoặ

c:\Users\ASUS\anaconda3\envs\easyocr_env\lib\site-packages\PIL\Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[OK] OCR đọc được:
HITACHI
Ảnh 25/391
Đang xử lý: http://fit.hcmute.edu.vn/Resources/Images/SubDomain/fit/20192020/hpt-logo.png
[OK] OCR đọc được:
knowing IT
Ảnh 26/391
Đang xử lý: http://fit.hcmute.edu.vn/Resources/Images/SubDomain/fit/20192020/techhorizon-logo.png
[!] Không đọc được chữ hoặc ảnh không có chữ.
Ảnh 27/391
Đang xử lý: http://fit.hcmute.edu.vn/Resources/Images/SubDomain/fit/20192020/triviet-logo.png
[!] Không đọc được chữ hoặc ảnh không có chữ.
Ảnh 28/391
Đang xử lý: http://fit.hcmute.edu.vn/Resources/Images/SubDomain/fit/20192020/ztxgrp_logo.jpg
[OK] OCR đọc được:
š
Ảnh 29/391
Đang xử lý: http://fit.hcmute.edu.vn/Resources/Images/SubDomain/fit/20192020/oracle-logo.png
[OK] OCR đọc được:
ORACT ESE
Ảnh 30/391
Đang xử lý: http://fit.hcmute.edu.vn/Resources/Images/SubDomain/fit/20192020/aws-logo.png
[OK] OCR đọc được:
lt

ine
Ảnh 31/391
Đang xử lý: http://fit.hcmute.edu.vn/Resources/Images/SubDomain/fit/20192020/kps-logo.png
[!] Không đọc được chữ hoặc ảnh không có chữ.
Ảnh